# Tutorial 5 — OMOP Validation

This notebook verifies that the harmonized outputs from Tutorial 4 conform to
OMOP CDM standards. Every check runs as a **federated metric or SQL query** —
raw data stays on the server; you receive only aggregate results.

**What does "passing" validation mean?**
It means the harmonized data is structurally correct, uses recognized OMOP
concept IDs, has no null values in required fields, and is internally consistent
across the three tables. It does not validate clinical correctness.

**Prerequisites:** Complete Tutorial 4. Paste your harmonized dataset UIDs below.

## Configuration

In [ ]:
import os
import rhino_health as rh
from getpass import getpass
from rhino_health import ApiEnvironment
from rhino_health.lib.metrics import Count, Mean

# From Tutorial 4 — paste your harmonized dataset UIDs here
PROJECT_UID         = "<YOUR-PROJECT-UID>"                  # REPLACE with your project UID
OMOP_PERSON_UID     = '<YOUR-OMOP-PERSON-DATASET-UID>'      # REPLACE with your OMOP Person dataset UID
OMOP_VISIT_UID      = '<YOUR-OMOP-VISIT-DATASET-UID>'       # REPLACE with your OMOP Visit dataset UID
OMOP_PROCEDURE_UID  = '<YOUR-OMOP-PROCEDURE-DATASET-UID>'   # REPLACE with your OMOP Procedure dataset UID

# Valid OMOP concept IDs for each vocabulary.
# 0 = "No matching concept" — allowed but flagged as a coverage warning.
VALID_GENDER_CONCEPTS    = {8507, 8532, 8521, 0}
VALID_RACE_CONCEPTS      = {8515, 8516, 8527, 8522, 0}
VALID_ETHNICITY_CONCEPTS = {38003563, 38003564, 0}
VALID_VISIT_CONCEPTS     = {9201, 9202, 9203, 0}
VALID_PROCEDURE_CONCEPTS = {4287782, 4196867, 4098498, 4098460, 4098462, 4129922, 0}

## Shared Utilities

In [ ]:
results = []  # accumulates (check_name, status, detail) tuples

def check(name, passed, detail="", warn_only=False):
    """Record and print a single validation result."""
    if passed:
        status = "PASS"
        icon   = "✅"
    elif warn_only:
        status = "WARN"
        icon   = "⚠️ "
    else:
        status = "FAIL"
        icon   = "❌"
    results.append((name, status, detail))
    print(f"  {icon} {name}")
    if detail:
        print(f"       → {detail}")

def safe_count(session, dataset_uid, col):
    """Return non-null count for col, or None on error."""
    try:
        return session.dataset.get_dataset_metric(
            dataset_uid, Count(variable=col)
        ).output.get("count", 0)
    except Exception as e:
        print(f"  (count unavailable for {col}: {e})")
        return None

def safe_histogram(session, dataset_uid, column):
    """Return histogram dict {str_value: count} via Count+group_by, or {} on error."""
    try:
        result = session.dataset.get_dataset_metric(
            dataset_uid,
            Count(variable=column, group_by={"groupings": [column]}),
        )
        raw = result.output
        if not raw:
            return {}
        first_val = next(iter(raw.values()))
        if isinstance(first_val, dict):
            return {k: v.get("count", 0) for k, v in raw.items()}
        return dict(raw)
    except Exception as e:
        print(f"  (histogram unavailable for {column}: {e})")
        return {}

def safe_schema_fields(session, data_schema_uid):
    """Return set of field names for a schema, or None on error."""
    if not data_schema_uid:
        return None
    try:
        schema = session.data_schema.get_data_schemas([data_schema_uid])[0]
        return set(schema.schema_fields.field_names)
    except Exception as e:
        print(f"  (schema fields unavailable: {e})")
        return None

print("Utilities loaded.")

## Authenticate and Load Datasets

In [ ]:
my_username = "stephanie+pla@rhinohealth.com"  # REPLACE
session = rh.login(username=my_username, password=getpass(), rhino_api_url=ApiEnvironment.PROD_AWS_URL)
print(f"Logged in as: {my_username}")

person_ds    = session.dataset.get_dataset(OMOP_PERSON_UID)
visit_ds     = session.dataset.get_dataset(OMOP_VISIT_UID)
procedure_ds = session.dataset.get_dataset(OMOP_PROCEDURE_UID)

# Map each dataset to its primary ID column for row counts
_id_cols = {
    OMOP_PERSON_UID:    "person_id",
    OMOP_VISIT_UID:     "visit_occurrence_id",
    OMOP_PROCEDURE_UID: "procedure_occurrence_id",
}

print(f"\n{'Dataset':<45} {'Rows':>8}")
print("─" * 55)
for ds in [person_ds, visit_ds, procedure_ds]:
    result = session.dataset.get_dataset_metric(ds.uid, Count(variable=_id_cols[ds.uid]))
    n = result.output.get("count", result.output)
    print(f"{ds.name:<45} {n:>8,}")

---
## Section 1 — OMOP Person: Structural Checks

**What we check:** Required OMOP columns are present; key fields have no nulls.

In OMOP CDM, `person_id` must be non-null and unique for every patient.
`gender_concept_id`, `race_concept_id`, and `year_of_birth` are required.

In [ ]:
print("=" * 55)
print("  OMOP Person — Structural Checks")
print("=" * 55)

required_cols = {"person_id", "gender_concept_id", "race_concept_id",
                 "ethnicity_concept_id", "year_of_birth"}

present_cols = safe_schema_fields(session, person_ds.data_schema_uid)
if present_cols is not None:
    missing = required_cols - present_cols
    check("Required OMOP Person columns present",
          len(missing) == 0,
          detail=f"Missing: {missing}" if missing else "")
else:
    print("  (schema not attached — skipping column presence check)")
    present_cols = required_cols  # assume all present, attempt metric checks anyway

_total = safe_count(session, person_ds.uid, "person_id")
if _total is not None:
    for col in sorted(required_cols):
        if col not in present_cols:
            continue
        non_null = safe_count(session, person_ds.uid, col)
        if non_null is None:
            check(f"No nulls in {col}", True, "metric unavailable — skipping", warn_only=True)
        else:
            null_rate = (_total - non_null) / _total if _total > 0 else 0
            check(f"No nulls in {col}", null_rate == 0.0,
                  detail=f"{null_rate*100:.1f}% null" if null_rate > 0 else "")
else:
    print("  (total count unavailable — skipping null checks)")

## Section 2 — OMOP Person: Concept ID Validity

**What we check:** All concept IDs use recognized OMOP values.
An unrecognized concept ID means the semantic mapping produced a wrong ID,
or a source value was not covered by the mapping.

We also separately flag records with `concept_id = 0` (the OMOP "no matching
concept" sentinel). These are technically valid but indicate incomplete coverage.

In [ ]:
print("=" * 55)
print("  OMOP Person — Concept ID Validity")
print("=" * 55)

for col, valid_set, label in [
    ("gender_concept_id",    VALID_GENDER_CONCEPTS,    "Gender"),
    ("race_concept_id",      VALID_RACE_CONCEPTS,      "Race"),
    ("ethnicity_concept_id", VALID_ETHNICITY_CONCEPTS, "Ethnicity"),
]:
    hist = safe_histogram(session, person_ds.uid, col)
    if not hist:
        check(f"{label} concept IDs valid", False, "Could not retrieve histogram")
        continue

    found   = {int(k) for k in hist}
    invalid = found - valid_set
    total   = sum(hist.values())
    zero_ct = hist.get("0", hist.get(0, 0))
    coverage = (total - zero_ct) / total * 100 if total else 0

    check(f"{label} concept IDs all recognized",
          len(invalid) == 0,
          detail=f"Unrecognized IDs found: {invalid}" if invalid else "")

    check(f"{label} mapping coverage ≥ 80%",
          coverage >= 80,
          detail=f"{coverage:.1f}% mapped to non-zero concept",
          warn_only=(60 <= coverage < 80))

    # Print distribution for reference
    concept_names = {
        8507: "Male", 8532: "Female", 8521: "Other/Unknown",
        8515: "Asian", 8516: "Black", 8527: "White", 8522: "Other Race",
        38003563: "Hispanic", 38003564: "Non-Hispanic", 0: "Unmapped (0)",
    }
    print(f"\n    {label} concept distribution:")
    for cid, cnt in sorted(hist.items(), key=lambda x: -x[1]):
        name = concept_names.get(int(cid), f"concept {cid}")
        print(f"      {int(cid):<10} {name:<30} {cnt:,}")

---
## Section 3 — OMOP Visit Occurrence: Structural & Concept Checks

In [ ]:
print("=" * 55)
print("  OMOP Visit Occurrence — Structural Checks")
print("=" * 55)

required_cols = {"visit_occurrence_id", "person_id", "visit_concept_id", "visit_start_date"}

present_cols = safe_schema_fields(session, visit_ds.data_schema_uid)
if present_cols is not None:
    missing = required_cols - present_cols
    check("Required OMOP Visit columns present",
          len(missing) == 0,
          detail=f"Missing: {missing}" if missing else "")
else:
    print("  (schema not attached — skipping column presence check)")
    present_cols = required_cols

_total = safe_count(session, visit_ds.uid, "visit_occurrence_id")
if _total is not None:
    for col in sorted(required_cols):
        if col not in present_cols:
            continue
        non_null = safe_count(session, visit_ds.uid, col)
        if non_null is None:
            check(f"No nulls in {col}", True, "metric unavailable — skipping", warn_only=True)
        else:
            null_rate = (_total - non_null) / _total if _total > 0 else 0
            check(f"No nulls in {col}", null_rate == 0.0,
                  detail=f"{null_rate*100:.1f}% null" if null_rate > 0 else "")
else:
    print("  (total count unavailable — skipping null checks)")

print()
hist = safe_histogram(session, visit_ds.uid, "visit_concept_id")
if hist:
    found   = {int(k) for k in hist}
    invalid = found - VALID_VISIT_CONCEPTS
    check("visit_concept_id values recognized",
          len(invalid) == 0,
          detail=f"Unrecognized IDs: {invalid}" if invalid else "")

    visit_names = {9201: "Inpatient", 9202: "Outpatient", 9203: "Emergency", 0: "Unmapped (0)"}
    print(f"\n    Visit type distribution:")
    for cid, cnt in sorted(hist.items(), key=lambda x: -x[1]):
        name = visit_names.get(int(cid), f"concept {cid}")
        print(f"      {int(cid):<8} {name:<15} {cnt:,}")

---
## Section 4 — OMOP Procedure Occurrence: Structural & Coverage Checks

In [ ]:
print("=" * 55)
print("  OMOP Procedure Occurrence — Structural Checks")
print("=" * 55)

required_cols = {"procedure_occurrence_id", "person_id", "procedure_concept_id",
                 "procedure_date", "visit_occurrence_id"}

present_cols = safe_schema_fields(session, procedure_ds.data_schema_uid)
if present_cols is not None:
    missing = required_cols - present_cols
    check("Required OMOP Procedure columns present",
          len(missing) == 0,
          detail=f"Missing: {missing}" if missing else "")
else:
    print("  (schema not attached — skipping column presence check)")
    present_cols = required_cols

_total = safe_count(session, procedure_ds.uid, "procedure_occurrence_id")
if _total is not None:
    for col in sorted(required_cols):
        if col not in present_cols:
            continue
        non_null = safe_count(session, procedure_ds.uid, col)
        if non_null is None:
            check(f"No nulls in {col}", True, "metric unavailable — skipping", warn_only=True)
        else:
            null_rate = (_total - non_null) / _total if _total > 0 else 0
            check(f"No nulls in {col}", null_rate == 0.0,
                  detail=f"{null_rate*100:.1f}% null" if null_rate > 0 else "")
else:
    print("  (total count unavailable — skipping null checks)")

print()
hist = safe_histogram(session, procedure_ds.uid, "procedure_concept_id")
if hist:
    found   = {int(k) for k in hist}
    invalid = found - VALID_PROCEDURE_CONCEPTS
    check("procedure_concept_id values recognized",
          len(invalid) == 0,
          detail=f"Unrecognized IDs: {invalid}" if invalid else "")

    total   = sum(hist.values())
    zero_ct = hist.get("0", hist.get(0, 0))
    cov     = (total - zero_ct) / total * 100 if total else 0
    check("Procedure mapping coverage ≥ 80%", cov >= 80,
          detail=f"{cov:.1f}% of procedures mapped to non-zero concept",
          warn_only=(60 <= cov < 80))

    proc_names = {
        4287782: "Colonoscopy", 4196867: "Appendectomy",
        4098498: "Office visit (new)", 4098460: "Office visit (est)",
        4098462: "Office visit (est)", 4129922: "ED visit", 0: "Unmapped (0)",
    }
    print(f"\n    Procedure concept distribution:")
    for cid, cnt in sorted(hist.items(), key=lambda x: -x[1]):
        name = proc_names.get(int(cid), f"concept {cid}")
        print(f"      {int(cid):<10} {name:<30} {cnt:,}")

---
## Section 5 — Referential Integrity

**What we check:** Do the IDs linking the three OMOP tables actually match up?
In OMOP, `person_id` in `visit_occurrence` and `procedure_occurrence` must
refer to a real patient in `person`. `visit_occurrence_id` in
`procedure_occurrence` must refer to a real visit.

We approximate these checks using federated SQL COUNT(DISTINCT ...) queries.

In [ ]:
print("=" * 55)
print("  Referential Integrity")
print("=" * 55)

def count_distinct(session, dataset_uid, col):
    """Approximate COUNT(DISTINCT col) by counting unique groups via Count + group_by."""
    result = session.dataset.get_dataset_metric(
        dataset_uid,
        Count(variable=col, group_by={"groupings": [col]}),
    )
    return len(result.output)

try:
    n_persons       = count_distinct(session, OMOP_PERSON_UID,    "person_id")
    n_visit_persons = count_distinct(session, OMOP_VISIT_UID,     "person_id")
    n_proc_persons  = count_distinct(session, OMOP_PROCEDURE_UID, "person_id")
    n_visits        = count_distinct(session, OMOP_VISIT_UID,     "visit_occurrence_id")
    n_proc_visits   = count_distinct(session, OMOP_PROCEDURE_UID, "visit_occurrence_id")

    check("visit_occurrence person_ids ≤ person table count",
          n_visit_persons <= n_persons,
          detail=f"{n_visit_persons} distinct in visits vs {n_persons} in person table")

    check("procedure_occurrence person_ids ≤ person table count",
          n_proc_persons <= n_persons,
          detail=f"{n_proc_persons} distinct in procedures vs {n_persons} in person table")

    check("procedure_occurrence visit_ids ≤ visit_occurrence count",
          n_proc_visits <= n_visits,
          detail=f"{n_proc_visits} distinct in procedures vs {n_visits} in visit table")

except Exception as e:
    print(f"  Referential integrity checks skipped: {e}")

---
## Validation Summary

In [ ]:
passes = [r for r in results if r[1] == "PASS"]
warns  = [r for r in results if r[1] == "WARN"]
fails  = [r for r in results if r[1] == "FAIL"]

print("=" * 65)
print(f"  OMOP Validation Summary — {len(results)} checks")
print("=" * 65)
print(f"  ✅ PASS:  {len(passes)}")
print(f"  ⚠️  WARN:  {len(warns)}")
print(f"  ❌ FAIL:  {len(fails)}")

if warns:
    print(f"\n  Warnings to investigate:")
    for name, _, detail in warns:
        print(f"    • {name}")
        if detail:
            print(f"        {detail}")

if fails:
    print(f"\n  Failures to resolve before using data:")
    for name, _, detail in fails:
        print(f"    • {name}")
        if detail:
            print(f"        {detail}")

if not warns and not fails:
    print("\n  All checks passed.")
    print("  Data is ready for federated analytics and model training.")
else:
    print("""
  Recommended next steps:
  • FAIL: Return to Tutorial 4, correct the semantic or syntactic
    mapping, re-run harmonization, then re-run this notebook.
  • WARN on coverage: Review which source values were not included
    in the semantic mapping. Add them, re-run harmonization.
  • Use the FCP Dashboard → Harmonization section to inspect and
    edit mapping entries without running the full notebook again.
""")

print("=" * 65)
print("Dashboard: Projects → Datasets — inspect the three OMOP output datasets.")

You should see 25 Pass & 2 Fail

The failures are expected and indicate that there are 98 distinct person_ids in the visits & procedures tables, but only 95 distinct person_ids in the person table. 
This means that there are 3 person_ids missing, either due to issues with data intake, transformations, or data scrubbing